In [1]:
import ibis
import duckdb as db
import pandas as pd
from dataclasses import dataclass
import sys
sys.path.append("../")
from sql_ai_agent.parse_query import is_markdown_code_chunk, extract_code_from_markdown
from sql_ai_agent2.db_handler import get_postgres_schema, get_duckdb_schema

In [2]:
tbl_name = "air_traffic"

In [3]:
con = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

In [4]:
print(type(con))
isinstance(con, ibis.backends.postgres.Backend)
con.get_schema(tbl_name)



<class 'ibis.backends.postgres.Backend'>


ibis.Schema {
  Activity Period              int64
  Activity Period Start Date   timestamp(6)
  Operating Airline            string
  Operating Airline IATA Code  string
  Published Airline            string
  Published Airline IATA Code  string
  GEO Summary                  string
  GEO Region                   string
  Activity Type Code           string
  Price Category Code          string
  Terminal                     string
  Boarding Area                string
  Passenger Count              int64
  data_as_of                   string
  data_loaded_at               string
}

In [9]:
if isinstance(con, ibis.backends.postgres.Backend):
    schema = get_postgres_schema(con = con, tbl_name = tbl_name)
elif isinstance(con, db.duckdb.DuckDBPyConnection):
    schema = get_duckdb_schema(con = con, tbl_name = tbl_name)

schema

'Activity Period BIGINT, Activity Period Start Date TIMESTAMP_NS, Operating Airline VARCHAR, Operating Airline IATA Code VARCHAR, Published Airline VARCHAR, Published Airline IATA Code VARCHAR, GEO Summary VARCHAR, GEO Region VARCHAR, Activity Type Code VARCHAR, Price Category Code VARCHAR, Terminal VARCHAR, Boarding Area VARCHAR, Passenger Count BIGINT, data_as_of VARCHAR, data_loaded_at VARCHAR'

In [6]:
con.get_schema("air_traffic")

ibis.Schema {
  Activity Period              int64
  Activity Period Start Date   timestamp(6)
  Operating Airline            string
  Operating Airline IATA Code  string
  Published Airline            string
  Published Airline IATA Code  string
  GEO Summary                  string
  GEO Region                   string
  Activity Type Code           string
  Price Category Code          string
  Terminal                     string
  Boarding Area                string
  Passenger Count              int64
  data_as_of                   string
  data_loaded_at               string
}

In [7]:
air_traffic = con.sql("SELECT * FROM air_traffic LIMIT 100").execute()

air_traffic.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
1,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
2,199907,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
3,199907,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM
4,199907,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198,2025/09/20 01:01:11 PM,2025/09/22 03:10:03 PM


In [8]:
con = db.connect()
con.register(tbl_name, air_traffic)
type(con)

duckdb.duckdb.DuckDBPyConnection

In [6]:
import duckdb as db
tbl_name = "air_traffic"
db.sql(f"DESCRIBE SELECT * FROM {tbl_name};").df()
table_schema = db.sql(f"DESCRIBE SELECT * FROM {tbl_name};").df()
col_info = table_schema[["column_name", "column_type"]]
col_info

CatalogException: Catalog Error: Table with name air_traffic does not exist!
Did you mean "pg_constraint"?

In [14]:
schema_str = ", ".join(
        f"{name} {dtype}"
        for name, dtype in zip(col_info["column_name"], col_info["column_type"])
    )
col_names=col_info["column_name"].tolist()
col_types=col_info["column_type"].tolist()
tbl_schema=schema_str

print(col_names)
print(col_types)
print(tbl_schema)

['Activity Period', 'Activity Period Start Date', 'Operating Airline', 'Operating Airline IATA Code', 'Published Airline', 'Published Airline IATA Code', 'GEO Summary', 'GEO Region', 'Activity Type Code', 'Price Category Code', 'Terminal', 'Boarding Area', 'Passenger Count', 'data_as_of', 'data_loaded_at']
['BIGINT', 'TIMESTAMP_NS', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'VARCHAR']
Activity Period BIGINT, Activity Period Start Date TIMESTAMP_NS, Operating Airline VARCHAR, Operating Airline IATA Code VARCHAR, Published Airline VARCHAR, Published Airline IATA Code VARCHAR, GEO Summary VARCHAR, GEO Region VARCHAR, Activity Type Code VARCHAR, Price Category Code VARCHAR, Terminal VARCHAR, Boarding Area VARCHAR, Passenger Count BIGINT, data_as_of VARCHAR, data_loaded_at VARCHAR


In [22]:
import sql_ai_agent2.prompt_handler as ph
con = db.connect()
con.register("air_traffic", air_traffic)
# db.register("air_traffic", air_traffic)
print(ph.get_tbl_attr(tbl_name = tbl_name))
type(con)

TableAttributes(col_names=['Activity Period', 'Activity Period Start Date', 'Operating Airline', 'Operating Airline IATA Code', 'Published Airline', 'Published Airline IATA Code', 'GEO Summary', 'GEO Region', 'Activity Type Code', 'Price Category Code', 'Terminal', 'Boarding Area', 'Passenger Count', 'data_as_of', 'data_loaded_at'], col_types=['BIGINT', 'TIMESTAMP_NS', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'BIGINT', 'VARCHAR', 'VARCHAR'], tbl_schema='Activity Period BIGINT, Activity Period Start Date TIMESTAMP_NS, Operating Airline VARCHAR, Operating Airline IATA Code VARCHAR, Published Airline VARCHAR, Published Airline IATA Code VARCHAR, GEO Summary VARCHAR, GEO Region VARCHAR, Activity Type Code VARCHAR, Price Category Code VARCHAR, Terminal VARCHAR, Boarding Area VARCHAR, Passenger Count BIGINT, data_as_of VARCHAR, data_loaded_at VARCHAR')


duckdb.duckdb.DuckDBPyConnection

In [ ]:
if isinstance(con, db.DuckDBPyConnection):
    if tbl_name in con.sql("SHOW TABLES").df()["name"].tolist():
        print(True)
    else:
        print(False)
elif isinstance(con, ibis.backends.postgres.Backend)


True


In [4]:
schema = con.get_schema("air_traffic")
schema.names

df = pd.DataFrame({
    'column_name': schema.names,
    'data_type': [str(dtype) for dtype in schema.types]
})


df

,column_name,data_type
0,Activity Period,int64
1,Activity Period Start Date,timestamp(6)
2,Operating Airline,string
3,Operating Airline IATA Code,string
4,Published Airline,string
5,Published Airline IATA Code,string
6,GEO Summary,string
7,GEO Region,string
8,Activity Type Code,string
9,Price Category Code,string


In [5]:
type(con)

ibis.backends.postgres.Backend

In [5]:
table_name = "air_traffic"

def get_schema(con, tbl_name: str):
    """
    Retrieve the schema information for a given PostgreSQL table.

    This function queries the `information_schema.columns` view to 
    return column-level metadata, including column names, data types,
    character lengths, and numeric precision details.

    Parameters
    ----------
    con : ibis.Connection
        An active Ibis connection to a PostgreSQL database.
    table_name : str
        Name of the table to retrieve schema information for.

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing one row per column, with the following fields:
        - column_name
        - data_type
        - character_maximum_length
        - numeric_precision
        - numeric_scale

    Notes
    -----
    - The function assumes `con.con` exposes a compatible DB-API connection
      that supports `pd.read_sql()`.
    - Use parameterized queries if `table_name` comes from user input to
      prevent SQL injection.
    """

    query = f"""
        SELECT 
            column_name,
            data_type,
            character_maximum_length,
            numeric_precision,
            numeric_scale
        FROM information_schema.columns
        WHERE table_name = '{tbl_name}'
        ORDER BY ordinal_position
    """
    df = pd.read_sql(query, con.con) 
    df = df.rename(columns={"data_type": "column_type"})

    return df


@dataclass
class TableAttributes:
    col_names: list[str]
    col_types: list[str]
    tbl_schema: str

def get_tbl_attr(con, tbl_name: str) -> TableAttributes:
    """
    Get column names, types, and schema definition string for a DuckDB table.
    """

    # Query schema
    table_schema = get_schema(con = con, tbl_name = tbl_name)
    col_info = table_schema[["column_name", "column_type"]]

    # Build schema string
    schema_str = ", ".join(
        f"{name} {dtype}"
        for name, dtype in zip(col_info["column_name"], col_info["column_type"])
    )

    return TableAttributes(
        col_names=col_info["column_name"].tolist(),
        col_types=col_info["column_type"].tolist(),
        tbl_schema=schema_str,
    )



In [7]:
schema = con.get_schema("air_traffic")
print(type(schema))
table_schema = schema.to_pandas()

table_schema

<class 'ibis.expr.schema.Schema'>


[('Activity Period', dtype('int64')),
 ('Activity Period Start Date', dtype('<M8[ns]')),
 ('Operating Airline', dtype('O')),
 ('Operating Airline IATA Code', dtype('O')),
 ('Published Airline', dtype('O')),
 ('Published Airline IATA Code', dtype('O')),
 ('GEO Summary', dtype('O')),
 ('GEO Region', dtype('O')),
 ('Activity Type Code', dtype('O')),
 ('Price Category Code', dtype('O')),
 ('Terminal', dtype('O')),
 ('Boarding Area', dtype('O')),
 ('Passenger Count', dtype('int64')),
 ('data_as_of', dtype('O')),
 ('data_loaded_at', dtype('O'))]

In [6]:
tbl_attr = get_tbl_attr(con = con, tbl_name = "air_traffic")
tbl_attr.tbl_schema


/tmp/ipykernel_929/4051372798.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con.con)


'Activity Period bigint, Activity Period Start Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint, data_as_of character varying, data_loaded_at character varying'

In [7]:
@dataclass
class SystemPrompt:
    system: str
    schema: str
    col_names: str
    col_types: str
    tbl_name: str


def system_prompt(tbl_name):
    # Get table schema
    tbl_attr = get_tbl_attr(tbl_name=tbl_name)

    # Prompt templates
    system_template = (
        "Given the following SQL table, your job is to write queries given a user’s request. "
        "Return just the SQL query as plain text, without additional text, and don't use markdown format.\n\n"
        f"CREATE TABLE {tbl_name} ({tbl_attr.tbl_schema})\n"
    )
    return SystemPrompt(
        system=system_template,
        schema=tbl_attr.tbl_schema,
        col_names=tbl_attr.col_names,
        col_types=tbl_attr.col_types,
        tbl_name=tbl_name,
    )


def user_prompt(question):
    user_template = f"Write a SQL query that returns: {question}"
    return user_template


In [8]:
from openai import OpenAI
class SqlAgent:
    def __init__(self, con, api_key, base_url, model, tbl_name, max_token=5000):
        self.api_key = api_key
        self.base_url = base_url
        self.model = model
        self.tbl_name = tbl_name
        self.max_token = max_token

        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.system = system_prompt(tbl_name=tbl_name)

    def send_prompt(self, question):
        self.user = user_prompt(question=question)
        self.response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system.system},
                {"role": "user", "content": self.user},
            ],
            max_completion_tokens=self.max_token,
        )

        content = self.response.choices[0].message.content
        if is_markdown_code_chunk(text=content):
            query = extract_code_from_markdown(markdown_text=content)
        else:
            query = content

        self.query = query

    def ask_question(self, question, verbose=True):
        self.send_prompt(question=question)
        self.data = con.sql(self.query).execute()
        if verbose:
            print(self.query)
            print(self.data)